# Credit Risk Modelling | Phases 1–3
**Predicting default, explaining risk drivers, and stress-testing credit decisions**

This self-contained notebook fits logistic regression and gradient boosting on 19 financial/repayment variables in the UCI credit-card default dataset, evaluates an untouched holdout set, and examines reliability and illustrative decision costs.

**Data:** Taiwan, 2005. **Use:** educational modelling only, not a lending or underwriting system.


In [ ]:
# In Colab, run this cell first.
%pip -q install ucimlrepo scikit-learn pandas numpy matplotlib statsmodels

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             roc_curve, precision_recall_curve, confusion_matrix,
                             ConfusionMatrixDisplay)
from sklearn.calibration import CalibrationDisplay
from pathlib import Path

RANDOM_STATE = 42
Path('figures').mkdir(exist_ok=True)


## 1. Data acquisition and validation
UCI hosts the original dataset, and `ucimlrepo` downloads it when this cell runs. **ID is not a predictive variable.** We deliberately exclude sex, age, education and marriage to focus on financial behavior.

In [ ]:
ds = fetch_ucirepo(id=350)
X_raw = ds.data.features.copy()
y_raw = ds.data.targets.squeeze().astype(int)
print('Dataset:', X_raw.shape, '| Default rate:', f'{y_raw.mean():.2%}')
print('Features:', X_raw.columns.tolist())
assert len(X_raw) == 30000
assert set(y_raw.unique()) == {0, 1}

# The UCI loader may return coded names X1...X23 or descriptive labels.
# Explicitly select exactly the 19 financial / repayment variables.
if {'X1', 'X6', 'X23'}.issubset(X_raw.columns):
    financial_cols = ['X1'] + [f'X{i}' for i in range(6, 24)]
    feature_map = {'X1': 'Credit limit'}
    feature_map.update({f'X{i}': f'Repayment status M{i-5}' for i in range(6,12)})
    feature_map.update({f'X{i}': f'Bill amount M{i-11}' for i in range(12,18)})
    feature_map.update({f'X{i}': f'Payment amount M{i-17}' for i in range(18,24)})
else:
    financial_cols = [c for c in X_raw.columns if c == 'LIMIT_BAL'
                      or c.startswith(('PAY_', 'BILL_AMT', 'PAY_AMT'))]
    feature_map = {c: c for c in financial_cols}
assert len(financial_cols) == 19, f'Expected 19 financial features, found: {financial_cols}'
assert all(c in X_raw.columns for c in financial_cols)
X = X_raw[financial_cols].apply(pd.to_numeric, errors='coerce').reset_index(drop=True)
y = y_raw.reset_index(drop=True)
print('Model features:', financial_cols)
print('Missing values:', X.isna().sum().sum())
display(X.describe().T.round(2))

## 2. Train / validation / test split
We create disjoint stratified splits: 60% training, 20% validation, 20% final test. Validation selects the model and threshold. The test set remains untouched until the last step. A random split evaluates other customers from this 2005 cohort, **not** performance in later years.

In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=.20, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=.25, stratify=y_trainval, random_state=RANDOM_STATE)
print('Train / validation / test:', len(X_train), len(X_val), len(X_test))
print('Default rates:', [round(v.mean(), 4) for v in [y_train, y_val, y_test]])
assert not set(X_train.index) & set(X_val.index)
assert not set(X_train.index) & set(X_test.index)
assert not set(X_val.index) & set(X_test.index)

## 3. Fit two models
The preprocessing is fit *inside* each pipeline, so validation and test observations cannot influence training-time preprocessing. The logistic model is interpretable; boosting learns nonlinear interactions.

In [ ]:
models = {
    'Logistic regression': Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    'Gradient boosting': Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('model', HistGradientBoostingClassifier(
            max_iter=180, max_leaf_nodes=15, learning_rate=.06,
            l2_regularization=2.0, random_state=RANDOM_STATE))
    ])
}
for name, model in models.items():
    model.fit(X_train, y_train)

def evaluate(y_true, p):
    return {'ROC-AUC': roc_auc_score(y_true, p),
            'PR-AUC': average_precision_score(y_true, p),
            'Brier': brier_score_loss(y_true, p)}

val_probs = {name: model.predict_proba(X_val)[:, 1]
             for name, model in models.items()}
validation = pd.DataFrame({name: evaluate(y_val, probs)
                           for name, probs in val_probs.items()}).T
display(validation.round(4))
selected_name = validation['PR-AUC'].idxmax()
selected_model = models[selected_name]
print('Selected by validation PR-AUC:', selected_name)

## 4. Risk thresholds and a simple business trade-off
A probability of default is more useful than a binary guess. We choose a threshold on **validation only**. For illustration, a missed default costs 5 units and an unnecessary flag costs 1 unit. These are *not* estimated bank economics.

In [ ]:
cost_false_negative = 5.0
cost_false_positive = 1.0
thresholds = np.linspace(.05, .95, 181)
p_val = val_probs[selected_name]
rows = []
for t in thresholds:
    flag = (p_val >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, flag, labels=[0,1]).ravel()
    rows.append({'Threshold':t, 'Cost per customer':(cost_false_negative*fn + cost_false_positive*fp)/len(y_val),
                 'Recall':tp/(tp+fn), 'Precision':tp/(tp+fp) if tp+fp else 0, 'Flagged share':flag.mean()})
threshold_table = pd.DataFrame(rows)
best = threshold_table.loc[threshold_table['Cost per customer'].idxmin()]
threshold = float(best['Threshold'])
print('Illustrative threshold:', round(threshold,3))
display(threshold_table.sort_values('Cost per customer').head(5).round(4))

## 5. Final holdout evaluation
Only now do we score the untouched test set. The numbers printed below are the basis for the final GitHub report.

In [ ]:
test_p = selected_model.predict_proba(X_test)[:, 1]
test_metrics = evaluate(y_test, test_p)
result = pd.Series({**test_metrics, 'Threshold':threshold,
    'Actual default rate':float(y_test.mean()),
    'Average predicted default':float(test_p.mean())}, name=selected_name)
display(result.to_frame('Test result').round(4))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ConfusionMatrixDisplay.from_predictions(y_test, test_p >= threshold, ax=axes[0], colorbar=False)
axes[0].set_title('Test confusion matrix')
CalibrationDisplay.from_predictions(y_test, test_p, n_bins=10, strategy='quantile', ax=axes[1])
axes[1].set_title('Probability calibration')
fpr, tpr, _ = roc_curve(y_test, test_p)
axes[2].plot(fpr,tpr,label=f"AUC {test_metrics['ROC-AUC']:.3f}")
axes[2].plot([0,1],[0,1],ls='--',alpha=.4)
axes[2].set(xlabel='False positive rate',ylabel='True positive rate',title='Test ROC curve')
axes[2].legend()
plt.tight_layout()
plt.savefig('figures/test_model_diagnostics.png',dpi=170,bbox_inches='tight')
plt.show()
result.to_frame('Value').to_csv('test_metrics.csv')
print('Saved: figures/test_model_diagnostics.png and test_metrics.csv')

## 6. Interpretability: validation-set permutation importance

Permutation importance quantifies how much **validation PR-AUC deteriorates** when each feature is shuffled. Because payment history and bill/payment amounts can be correlated, importance may be shared across variables. This is **predictive association**, not evidence that changing a feature causes default risk to change.

**Important:** We run this analysis on validation records only. The model and decision threshold have already been chosen; do not use these plots to retune after inspecting test results.

In [ ]:
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

perm = permutation_importance(
    selected_model, X_val, y_val,
    scoring='average_precision', n_repeats=5,
    random_state=RANDOM_STATE, n_jobs=-1
)
importance = pd.DataFrame({
    'Feature': X_val.columns,
    'Description': [feature_map[x] for x in X_val.columns],
    'PR-AUC drop': perm.importances_mean,
    'Std across shuffles': perm.importances_std
}).sort_values('PR-AUC drop', ascending=False)
display(importance.round(4))
importance.to_csv('feature_importance.csv', index=False)

show = importance.head(12).sort_values('PR-AUC drop')
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(show['Description'], show['PR-AUC drop'], xerr=show['Std across shuffles'], capsize=3)
ax.axvline(0, ls='--', color='gray', lw=1)
ax.set(xlabel='Decrease in validation PR-AUC when shuffled',
       title='Permutation importance | Validation set')
plt.tight_layout()
plt.savefig('figures/permutation_importance.png', dpi=170, bbox_inches='tight')
plt.show()

## 7. Partial dependence: repayment status and credit limit

Partial dependence plots show the **average model response** when a feature is set to a range of values across validation observations, holding other recorded features as they were. These are not causal effects and can create unrealistic feature combinations when inputs are correlated. The repayment-status values are coded, and their interpretation depends on the UCI data dictionary.

In [ ]:
# Prefer the latest repayment-status field and credit limit; both schemas supported.
repay_feature = 'X6' if 'X6' in X_val.columns else ('PAY_0' if 'PAY_0' in X_val.columns else 'PAY_1')
credit_feature = 'X1' if 'X1' in X_val.columns else 'LIMIT_BAL'

# Display partial dependence over observed validation values, not the final test set.
pdp_sample = X_val.sample(n=min(len(X_val), 2000), random_state=RANDOM_STATE)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, feature in zip(axes, [repay_feature, credit_feature]):
    PartialDependenceDisplay.from_estimator(
        selected_model, pdp_sample, features=[feature],
        response_method='predict_proba', kind='average',
        percentiles=(.05, .95), grid_resolution=25, ax=ax
    )
    ax.set_title(feature_map[feature])
    ax.set_ylabel('Average predicted default probability')
plt.suptitle('Partial dependence | Validation subset')
plt.tight_layout()
plt.savefig('figures/partial_dependence.png', dpi=170, bbox_inches='tight')
plt.show()

## 8. Holdout segment diagnostics (descriptive only)

Examine model performance across credit-limit bands and latest repayment-status groups. Test results are **reported**, not used to select a different model or threshold. Small categories may have unstable estimates; inspect counts before interpreting differences.

In [ ]:
# Use previously computed untouched holdout predictions, without refitting or retuning.
segments = X_test[[credit_feature, repay_feature]].copy()
segments['Actual default'] = y_test.to_numpy()
segments['Predicted PD'] = test_p
segments['Flagged'] = (test_p >= threshold).astype(int)
segments['Credit limit quartile'] = pd.qcut(segments[credit_feature], 4, duplicates='drop').astype(str)
segments['Latest repayment status'] = segments[repay_feature].astype('Int64').astype(str)

def segment_summary(data, group):
    rows = []
    for name, g in data.groupby(group, dropna=False, observed=True):
        rows.append({'Segment': str(name), 'Customers': len(g),
                     'Actual default rate': g['Actual default'].mean(),
                     'Average predicted PD': g['Predicted PD'].mean(),
                     'Flagged fraction': g['Flagged'].mean(),
                     'ROC-AUC': (roc_auc_score(g['Actual default'], g['Predicted PD'])
                                 if g['Actual default'].nunique() == 2 else np.nan)})
    return pd.DataFrame(rows)

by_limit = segment_summary(segments, 'Credit limit quartile')
by_repay = segment_summary(segments, 'Latest repayment status')
print('Credit limit segments:')
display(by_limit.round(3))
print('Latest repayment-status segments:')
display(by_repay.round(3))
by_limit.to_csv('credit_limit_segments.csv', index=False)
by_repay.to_csv('repayment_status_segments.csv', index=False)

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, table, title in [(axs[0], by_limit, 'Credit limit'), (axs[1], by_repay, 'Latest repayment status')]:
    pos = np.arange(len(table))
    ax.bar(pos - .18, 100 * table['Actual default rate'], width=.36, label='Observed')
    ax.bar(pos + .18, 100 * table['Average predicted PD'], width=.36, label='Predicted')
    ax.set_xticks(pos, table['Segment'], rotation=45, ha='right')
    ax.set_ylabel('Default rate / predicted PD (%)')
    ax.set_title(title)
    ax.legend()
plt.suptitle('Default rates by segment | Holdout test')
plt.tight_layout()
plt.savefig('figures/segment_calibration.png', dpi=170, bbox_inches='tight')
plt.show()

## 9. Interpretation checklist

After running, write down:

1. The three most important features **by permutation PR-AUC drop**.
2. Whether higher repayment delays correspond to higher **model-predicted** PD in partial dependence.
3. Which credit-limit or repayment-status segments show the largest observed vs. predicted differences.
4. Whether sample sizes are sufficient to trust those segment comparisons.
5. Why this is not a causal study, a time-based validation, or a fair lending decision system.

### Next phase

Add temporal validation on data with reliable origination and outcome timestamps, uncertainty intervals for AUC/Brier, and an explicitly economic decision-policy analysis. Do not deploy this educational model for real lending decisions.

## 10. Repayment-status definitions and sparse groups

The UCI data dictionary explicitly defines **−1** as paying duly and **1 to 9** as the number of delayed months (9 means nine or more). The observed dataset also contains **−2** and **0**, which the official UCI definition does not explicitly explain. Some secondary references describe these as *no consumption* and *revolving credit*, respectively, but we treat these as **special codes**, not as negative or zero months of delinquency. The source also does not promise that every coded status appears in our holdout sample.

Very small groups can display extreme default rates such as 100% even when the model is not systematically wrong. We therefore report group sizes and **95% Wilson binomial intervals** for observed default frequencies, alongside predicted PD.

Source: [UCI dataset 350](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients).

In [ ]:
from statsmodels.stats.proportion import proportion_confint

def status_text(x):
    x = int(x)
    if x == -2: return "-2 (special code)"
    if x == -1: return "-1 (pay duly)"
    if x == 0: return "0 (special code)"
    return f"{x} (delay {x}+ month{'s' if x!=1 else ''})" if x == 9 else f"{x} (delay {x} month{'s' if x!=1 else ''})"

counts = segments.groupby("Latest repayment status", observed=True).agg(
    Customers=("Actual default", "size"),
    Defaults=("Actual default", "sum"),
    Observed_PD=("Actual default", "mean"),
    Mean_predicted_PD=("Predicted PD", "mean")
).reset_index()
counts["code"] = pd.to_numeric(counts["Latest repayment status"], errors="coerce")
counts = counts.sort_values("code")
lo, hi = proportion_confint(
    counts["Defaults"].to_numpy(),
    counts["Customers"].to_numpy(),
    alpha=.05, method="wilson"
)
counts["CI_low"], counts["CI_high"] = lo, hi
counts["Label"] = counts["code"].map(status_text)
counts["Absolute_gap_pp"] = 100 * (counts["Observed_PD"] - counts["Mean_predicted_PD"])
counts["Small_group"] = counts["Customers"] < 30
display(counts[["Label","Customers","Defaults","Observed_PD","Mean_predicted_PD",
                "CI_low","CI_high","Absolute_gap_pp","Small_group"]].round(3))
counts.to_csv("repayment_status_uncertainty.csv", index=False)

fig, ax = plt.subplots(figsize=(11,5))
x = np.arange(len(counts))
yerr = [counts["Observed_PD"]-counts["CI_low"],
        counts["CI_high"]-counts["Observed_PD"]]
ax.errorbar(x-.10, counts["Observed_PD"], yerr=yerr, fmt="o", capsize=4,
            label="Observed default rate (95% Wilson CI)")
ax.scatter(x+.10, counts["Mean_predicted_PD"], marker="s",
           label="Mean predicted PD")
ax.set_xticks(x, counts["code"].astype(int).astype(str))
ax.set(xlabel="Most recent repayment-status code",ylabel="Default rate / predicted PD",
       title="Segment calibration with uncertainty | Untouched test set")
ax.set_ylim(0, 1.08)
ax.legend(loc="upper left")
for i, row in enumerate(counts.itertuples()):
    ax.text(i, 1.025, f"n={row.Customers}", ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("figures/repayment_status_confidence_intervals.png",dpi=170,bbox_inches="tight")
plt.show()
print("Segments with fewer than 30 customers:", counts.loc[counts.Small_group,"Label"].tolist())

## 11. Test-set uncertainty: bootstrap intervals

An AUC or Brier score is an estimate, not a known population parameter. We use stratified bootstrap resampling **only to quantify uncertainty**, without selecting models, features, or thresholds from the holdout test set. These intervals are descriptive and depend on the IID sampling assumption.

In [ ]:
rng_boot = np.random.default_rng(RANDOM_STATE)
truth = y_test.to_numpy()
pos = np.flatnonzero(truth == 1)
neg = np.flatnonzero(truth == 0)
boot = []
for _ in range(400):
    ix = np.concatenate([rng_boot.choice(pos, len(pos), replace=True),
                         rng_boot.choice(neg, len(neg), replace=True)])
    boot.append([roc_auc_score(truth[ix], test_p[ix]),
                 average_precision_score(truth[ix], test_p[ix]),
                 brier_score_loss(truth[ix], test_p[ix])])
boot = np.asarray(boot)
interval = pd.DataFrame({
    "Metric": ["ROC-AUC","PR-AUC","Brier"],
    "Observed": [test_metrics["ROC-AUC"], test_metrics["PR-AUC"], test_metrics["Brier"]],
    "CI_low":np.quantile(boot,.025,axis=0),
    "CI_high":np.quantile(boot,.975,axis=0)
})
display(interval.round(4))
interval.to_csv("model_metric_uncertainty.csv",index=False)

## 12. Decision thresholds and illustrative costs

A PD estimate alone does not determine the right lending action. We compare thresholds using **hypothetical cost units**, not real dollars or calibrated loss given default (LGD). Here, missing a default costs 5 units, while flagging a non-default costs 1 unit. These are teaching assumptions, **not bank lending economics**.

The threshold was already chosen on the validation set in Section 4. The test-set curves below are **descriptive only** and must not be used to reselect the threshold. A real credit decision would also consider exposure, LGD, profitability, customer outcomes and regulation.

In [ ]:
cost_scenarios = {"FN:FP = 2:1":(2.0,1.0),
                  "FN:FP = 5:1":(5.0,1.0),
                  "FN:FP = 10:1":(10.0,1.0)}
def threshold_metrics(y_true, probabilities, t, fn_cost, fp_cost):
    pred = probabilities >= t
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    return {"Threshold":t, "TP":tp,"FP":fp,"FN":fn,"TN":tn,
            "Recall":tp/(tp+fn) if tp+fn else np.nan,
            "Precision":tp/(tp+fp) if tp+fp else np.nan,
            "Flagged_share":pred.mean(),
            "Cost_per_customer":(fn_cost*fn+fp_cost*fp)/len(y_true)}
val_selected = threshold_metrics(y_val.to_numpy(),p_val,threshold,5.,1.)
holdout_selected = threshold_metrics(truth,test_p,threshold,5.,1.)
print("Threshold selected on validation:",round(threshold,3))
display(pd.DataFrame([{"Set":"Validation",**val_selected},
                      {"Set":"Holdout",**holdout_selected}]).round(4))

descriptive_rows = []
for label, (cost_fn,cost_fp) in cost_scenarios.items():
    for t in np.linspace(.05,.95,91):
        descriptive_rows.append({"Scenario":label,
                                 **threshold_metrics(truth,test_p,float(t),cost_fn,cost_fp)})
decision_table = pd.DataFrame(descriptive_rows)
fig,ax=plt.subplots(figsize=(9,5))
for scenario,group in decision_table.groupby("Scenario"):
    ax.plot(group["Threshold"],group["Cost_per_customer"],label=scenario)
ax.axvline(threshold, ls="--", color="gray", label="Validation-chosen threshold")
ax.set(xlabel="Default probability threshold",ylabel="Illustrative cost units per customer",
       title="How decision costs depend on threshold | Holdout, descriptive only")
ax.legend()
plt.tight_layout()
plt.savefig("figures/threshold_cost_scenarios.png",dpi=170,bbox_inches="tight")
plt.show()
decision_table.to_csv("threshold_cost_scenarios.csv",index=False)

## 13. Reliability summary and final interpretation

- The **holdout test** is evaluated once for final performance; later plots are exploratory diagnostics, not additional model-selection rounds.
- Look closely at `Customers` and Wilson interval widths before interpreting extreme default rates for rare repayment codes.
- Repayment-status values **−2 and 0** are special codes; they should not be interpreted as consecutive months overdue.
- Permutation importance and partial dependence are **predictive associations**, not causes.
- The cost curves illustrate sensitivity to hypothetical relative losses. They are **not calibrated underwriting policies**.
- This 2005 Taiwan dataset is not a validation of current borrowers, modern credit portfolios or demographic fairness. No protected-class fairness audit is performed because demographic variables were excluded.
- Save the executed Colab notebook and generated `figures/` and `.csv` files before preparing the final GitHub package.